# Support Vector Machines (SVM) with Wine Dataset
---
In this notebook, you'll apply Support Vector Machines (SVMs) to a more complex classification problem: classifying different types of wine based on their chemical properties.

## Step 1: Dataset Overview
We'll be using the [Wine Dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html) from scikit-learn. It contains 178 samples of wine with 13 chemical analysis features. The target is to classify the wines into three different cultivars.

**Features include**: alcohol content, malic acid, ash, magnesium, total phenols, flavanoids, non-flavanoid phenols, proanthocyanins, color intensity, hue, etc.

**Goal**: Train an SVM to predict the correct wine class from these features.

## Step 2: Install Required Libraries
If necessary, install missing libraries using:
```bash
pip install pandas numpy seaborn matplotlib scikit-learn
```

## Step 3: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## Step 4: Load and Explore the Wine Dataset

In [ ]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name='target')
X['target'] = y
X.head()

## Step 5: Prepare Data (Train/Validation Split)
We will split the dataset into training (70%) and validation (30%) sets, and scale the features.

In [ ]:
# Separate features and target
X = X.drop(columns=['target'])

# Split into train and validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

## Step 6: Train SVM with Linear Kernel

In [ ]:
svm_linear = SVC(kernel='linear')
svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_val_scaled)

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_linear))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_linear))

## Step 7: Train SVM with RBF Kernel

In [ ]:
svm_rbf = SVC(kernel='rbf')
svm_rbf.fit(X_train_scaled, y_train)
y_pred_rbf = svm_rbf.predict(X_val_scaled)

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_rbf))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_rbf))

## Step 8: Accuracy Comparison

In [ ]:
print(f"Linear Kernel Accuracy: {accuracy_score(y_val, y_pred_linear):.2f}")
print(f"RBF Kernel Accuracy: {accuracy_score(y_val, y_pred_rbf):.2f}")

## Investigation tasks

**Task 1**: Research and explain what a kernel is in SVMs and the different kernels that Sklearn supports.

<span style="color:red">(1 mark)</span>

A kernel is a function that transforms the input data to a higher-dimensional space, in doing so it allows the SVM to be able to find a hyperplane that can seperate the classes.

Sklearn supports the following kernels:
- Linear
- RBF (Radial Basis Function)
- Poly (Polynomial)
- Sigmoid
- Precomputed

**Task 2**: Train a SVM with a different kernel.

<span style="color:red">(1 mark)</span>

In [ ]:
svm_sigmoid = SVC(kernel='sigmoid')
svm_sigmoid.fit(X_train_scaled, y_train)
y_pred_sigmoid = svm_sigmoid.predict(X_val_scaled)

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred_rbf))
print("\nClassification Report:")
print(classification_report(y_val, y_pred_rbf))

**Task 3**: Generate a correlation matrix between all the features of the dataset and the target (you can use AI for this step), interpret the results and train a new SVM using only the 5 features with the highest correlation.

<span style="color:red">(1 mark)</span>

In [ ]:
# Add target back to compute correlations
X_with_target = X.copy()
X_with_target['target'] = y

# Correlation matrix
corr_matrix = X_with_target.corr()

# Plot heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix (Features + Target)')
plt.tight_layout()
plt.show()

# Find the 5 features most correlated with target (by absolute value)
target_corr = corr_matrix['target'].drop('target').abs().sort_values(ascending=False)
print("Feature correlations with target (absolute value):")
print(target_corr.round(3))

top5_features = target_corr.head(5).index.tolist()
print(f"\nTop 5 features: {top5_features}")

### Correlation Matrix Interpretation

The correlation matrix shows the pairwise linear correlation between all features and the target class (values range from -1 to +1).

**Top 5 features most correlated with the target** (by absolute correlation):

| Feature | Correlation | Interpretation |
|---|---|---|
| `flavanoids` | ~0.85 | Strong negative — lower flavanoids → higher class number |
| `od280/od315_of_diluted_wines` | ~0.79 | Strong negative — related to protein content |
| `total_phenols` | ~0.72 | Strong negative — phenol levels distinguish wine types |
| `color_intensity` | ~0.65 | Strong positive — darker wines tend to be a different cultivar |
| `proline` | ~0.63 | Strong positive — amino acid levels vary by cultivar |

**Key observations:**
- Several features have strong correlations with the target, meaning they carry significant class-discriminative information.
- Features like `flavanoids`, `total_phenols`, and `od280/od315` are highly intercorrelated with each other — they all relate to polyphenol content. Using all 13 features may include redundancy.
- By selecting only the top 5 features we reduce dimensionality while retaining the most informative signals, which can improve generalisation and reduce noise.

In [ ]:
# Train SVM (RBF) using only the top 5 features
X_train_top5 = X_train[top5_features]
X_val_top5 = X_val[top5_features]

scaler_top5 = StandardScaler()
X_train_top5_scaled = scaler_top5.fit_transform(X_train_top5)
X_val_top5_scaled = scaler_top5.transform(X_val_top5)

svm_top5 = SVC(kernel='rbf')
svm_top5.fit(X_train_top5_scaled, y_train)
y_pred_top5 = svm_top5.predict(X_val_top5_scaled)

print("\nConfusion Matrix (Top 5 Features):")
print(confusion_matrix(y_val, y_pred_top5))
print("\nClassification Report (Top 5 Features):")
print(classification_report(y_val, y_pred_top5))
print(f"Accuracy (Top 5 Features): {accuracy_score(y_val, y_pred_top5):.2f}")

**Task 4**: Compare the results of this new algorithm with the ones trained on steps 6 and 7.

<span style="color:red">(1 mark)</span>

The correlation SVM resulted in a validation acccuaracy of 0.98, compared against the 0.98 of the RBF SVM and the 0.96 of the Linear SVM. Showing how the correlation matrix did not show much benefit or loss over the RBF SVM trained on everything, as the accuracy was already so high, but against the linear SVM, which did not have the same flexibility to go into higher demensional space, there was a clear and distinct gap between them.

**Task 5**: Make predictions on the training set and compare the performance between the training and validation. Based on these results, determine if the model suffers from overfitting or underfitting.

<span style="color:red">(1 mark)</span>

In [ ]:
# Predict on training set for all three models
y_train_pred_linear = svm_linear.predict(X_train_scaled)
y_train_pred_rbf    = svm_rbf.predict(X_train_scaled)
y_train_pred_top5   = svm_top5.predict(X_train_top5_scaled)

models = {
    'Linear (all 13)': (y_train_pred_linear, y_pred_linear),
    'RBF (all 13)':    (y_train_pred_rbf,    y_pred_rbf),
    'RBF (top 5)':     (y_train_pred_top5,   y_pred_top5),
}

print(f"{'Model':<20} {'Train Acc':>10} {'Val Acc':>10} {'Gap':>8}")
print("-" * 52)
for name, (y_tr, y_v) in models.items():
    train_acc = accuracy_score(y_train, y_tr)
    val_acc   = accuracy_score(y_val,   y_v)
    print(f"{name:<20} {train_acc:>10.4f} {val_acc:>10.4f} {train_acc - val_acc:>8.4f}")

Due to the high accuracy scored on both data sets, there is some very minor overfitting on the training for linear and RBF all 13, showing 1 and 0.96 and 0.98 respectively. Furthermore the RBF top 5 model has no overfitting as the parameters for it is less. But overall, the models all perform very well, and  even in course of overfitting, it is still very minor.

**Bonus**: Research what feature scaling is and explain why it is important to apply it to this dataset as part of the data cleaning process.

<span style="color:red">(0.5 marks)</span>


**Bonus**: Research what PCA. Reduce dimensionality using PCA and visualize 2D scatter plots colored by class.

<span style="color:red">(0.5 marks)</span>